# Respiratory lung tracking — Colab demo

Fine-tunovani U-Net segmentira oba plućna krila u MP4 ili PNG sekvenci. Notebook automatski standardizuje kontrast/polaritet cele sekvence, prati levo i desno krilo odvojeno i u nastavku prikazuje video, grafik i tabelu merenja.

**Ulaz:** MP4, jedna PNG/JPG slika ili ZIP arhiva PNG frejmova.

In [ ]:
# Instalacija zavisnosti (Google Colab).
!pip -q install opencv-python-headless scipy pandas matplotlib

import os, sys, subprocess, zipfile, shutil
from pathlib import Path
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from scipy import ndimage
from scipy.signal import savgol_filter
from tqdm.auto import tqdm
from IPython.display import display, Video, Image

WORKDIR = Path('/content/respiratory_lung_tracking')
UPLOAD_DIR = WORKDIR / 'uploads'
OUTPUT_DIR = WORKDIR / 'outputs'
MODEL_PATH = WORKDIR / 'best_lung_unet.pt'
for folder in (WORKDIR, UPLOAD_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# Preuzimanje već fine-tunovanog modela iz ovog GitHub repozitorijuma.
MODEL_URL = 'https://raw.githubusercontent.com/markovich1803/respiratory-lung-tracking/main/best_lung_unet.pt'
if not MODEL_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print(f'Model ready: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Model arhitektura — identična modelu koji je fine-tunovan.
SIZE = 512

class Block(nn.Module):
    def __init__(self, a, b):
        super().__init__()
        self.x = nn.Sequential(
            nn.Conv2d(a, b, 3, 1, 1), nn.BatchNorm2d(b), nn.ReLU(),
            nn.Conv2d(b, b, 3, 1, 1), nn.BatchNorm2d(b), nn.ReLU(),
        )
    def forward(self, x):
        return self.x(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.p = nn.MaxPool2d(2)
        self.a, self.b, self.c, self.d, self.m = Block(1,32), Block(32,64), Block(64,128), Block(128,256), Block(256,512)
        self.u4, self.z4 = nn.ConvTranspose2d(512,256,2,2), Block(512,256)
        self.u3, self.z3 = nn.ConvTranspose2d(256,128,2,2), Block(256,128)
        self.u2, self.z2 = nn.ConvTranspose2d(128,64,2,2), Block(128,64)
        self.u1, self.z1 = nn.ConvTranspose2d(64,32,2,2), Block(64,32)
        self.o = nn.Conv2d(32,1,1)
    def forward(self, x):
        a=self.a(x); b=self.b(self.p(a)); c=self.c(self.p(b)); d=self.d(self.p(c)); m=self.m(self.p(d))
        d=self.z4(torch.cat((self.u4(m),d),1)); c=self.z3(torch.cat((self.u3(d),c),1))
        b=self.z2(torch.cat((self.u2(c),b),1)); a=self.z1(torch.cat((self.u1(b),a),1))
        return self.o(a)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
model = UNet().to(DEVICE)
model.load_state_dict(checkpoint['state'])
model.eval()
print('Model loaded. Best validation Dice:', f"{checkpoint.get('dice', float('nan')):.4f}")

In [ ]:
# Ručno učitavanje testa: MP4, PNG/JPG ili ZIP sa PNG/JPG frejmovima.
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Nije izabran test fajl.')

uploaded_name = next(iter(uploaded))
input_path = UPLOAD_DIR / uploaded_name
shutil.move(uploaded_name, input_path)
print('Uploaded:', input_path.name)

def load_frames(path):
    suffix = path.suffix.lower()
    if suffix == '.zip':
        sequence_dir = UPLOAD_DIR / f'{path.stem}_frames'
        sequence_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(path) as archive:
            archive.extractall(sequence_dir)
        paths = sorted(p for p in sequence_dir.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'})
        frames = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in paths]
        frames = [x for x in frames if x is not None]
        if not frames: raise RuntimeError('ZIP ne sadrži čitljive PNG/JPG frejmove.')
        return frames, None, path.stem
    if suffix in {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}:
        frame = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if frame is None: raise RuntimeError('Slika nije čitljiva.')
        return [frame], None, path.stem
    capture = cv2.VideoCapture(str(path))
    fps = float(capture.get(cv2.CAP_PROP_FPS)) or 12.0
    frames = []
    while True:
        ok, frame = capture.read()
        if not ok: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    capture.release()
    if not frames: raise RuntimeError('MP4 nema čitljive frejmove.')
    return frames, fps, path.stem

raw_frames, fps, sequence_name = load_frames(input_path)
print(f'Loaded {len(raw_frames)} frame(s); FPS: {fps if fps else 12.0}')

In [ ]:
# Standardizacija cele sekvence, segmentacija oba plućna krila i vizualizacija.
def standardize_polarity(x):
    h, w = x.shape
    y0, y1 = round(.25*h), round(.70*h)
    left = x[y0:y1, round(.16*w):round(.40*w)]
    right = x[y0:y1, round(.60*w):round(.84*w)]
    centre = x[round(.25*h):round(.75*h), round(.44*w):round(.56*w)]
    lungs = np.concatenate((left.ravel(), right.ravel()))
    return cv2.bitwise_not(x) if lungs.size and np.median(lungs) > np.median(centre) else x

def normalize_sequence(frames, border_fraction=.05):
    canonical = [standardize_polarity(x) for x in frames]
    stats = []
    for x in canonical:
        h,w=x.shape; dy=max(1,round(h*border_fraction)); dx=max(1,round(w*border_fraction))
        stats.append(np.percentile(x[dy:h-dy, dx:w-dx], (1,99)))
    stats = np.asarray(stats, dtype=np.float32)
    window = min(5, len(stats) if len(stats)%2 else len(stats)-1)
    if window >= 3: stats = ndimage.median_filter(stats, size=(window,1), mode='nearest')
    return [np.clip((x.astype(np.float32)-low)*255/max(high-low,1),0,255).astype(np.uint8) for x,(low,high) in zip(canonical,stats)]

def infer_mask(x):
    h,w=x.shape
    inp=cv2.resize(x,(SIZE,SIZE)).astype(np.float32)/255.0
    tensor=torch.from_numpy(inp[None,None]).to(DEVICE)
    with torch.inference_mode(): mask=(torch.sigmoid(model(tensor))[0,0].cpu().numpy()>=.5)
    labels,n=ndimage.label(mask)
    if n:
        sizes=np.asarray(ndimage.sum(mask,labels,range(1,n+1)))
        mask=np.isin(labels,np.argsort(sizes)[-min(2,n):]+1)
    return cv2.resize(mask.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)

def split_lungs(mask):
    labels,n=ndimage.label(mask); parts=[labels==label for label in range(1,n+1)]
    if len(parts)>=2:
        parts=sorted(parts,key=lambda item: np.nonzero(item)[1].mean())
        return parts[0],parts[-1]
    if len(parts)==1:
        _,xs=np.nonzero(parts[0]); midpoint=int(np.median(xs)); xx=np.arange(mask.shape[1])[None,:]
        return parts[0]&(xx<=midpoint),parts[0]&(xx>midpoint)
    return np.zeros_like(mask,dtype=bool),np.zeros_like(mask,dtype=bool)

def metrics(mask):
    y,_=np.nonzero(mask)
    if not len(y): return dict(apex_y=np.nan,bottom_y=np.nan,height_px=np.nan,area_px=0)
    top=int(y.min()); apex=float(y[y<=top+2].mean()); bottom=float(np.quantile(y,.98))
    return dict(apex_y=apex,bottom_y=bottom,height_px=bottom-apex,area_px=int(mask.sum()))

def visual_frame(x,left,right,index,fps):
    h,w=x.shape; original=cv2.cvtColor(x,cv2.COLOR_GRAY2BGR); overlay=original.copy(); binary=np.zeros_like(overlay)
    result=[]
    for mask,label,color in ((left,'LEFT',(255,180,0)),(right,'RIGHT',(0,220,0))):
        layer=np.full_like(overlay,color); overlay[mask]=cv2.addWeighted(overlay[mask],.5,layer[mask],.5,0); binary[mask]=color
        contour,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(overlay,contour,-1,color,max(1,round(w/240)))
        values=metrics(mask); result.append(values)
        if values['area_px']:
            x0,x1=(0,w//2-1) if label=='LEFT' else (w//2,w-1)
            for yy,kind in ((round(values['apex_y']),'APEX'),(round(values['bottom_y']),'BASE')):
                cv2.line(overlay,(x0,yy),(x1,yy),color,max(1,round(w/320)))
                cv2.putText(overlay,f'{label} {kind}',(x0+6,max(20,yy-7)),cv2.FONT_HERSHEY_SIMPLEX,.43,color,1,cv2.LINE_AA)
    for panel,title in zip((original,overlay,binary),('STANDARDIZED INPUT','LEFT + RIGHT LUNG SEGMENTATION','SEPARATE BINARY MASKS')):
        cv2.rectangle(panel,(0,0),(w,31),(0,0,0),-1); cv2.putText(panel,title,(10,22),cv2.FONT_HERSHEY_SIMPLEX,.6,(255,255,255),1,cv2.LINE_AA)
    out=cv2.hconcat((original,overlay,binary)); lm,rm=result; seconds=index/(fps or 12.0)
    text=f'Frame {index} | {seconds:.2f} s | Left: H {lm["height_px"]:.0f} px, A {lm["area_px"]} | Right: H {rm["height_px"]:.0f} px, A {rm["area_px"]}'
    cv2.rectangle(out,(0,h-34),(out.shape[1],h),(0,0,0),-1); cv2.putText(out,text,(10,h-11),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2,cv2.LINE_AA)
    return out, lm, rm

frames = normalize_sequence(raw_frames)
rows=[]; writer=None; mp4_path=OUTPUT_DIR/f'{sequence_name}_segmentation.mp4'
for index, frame in enumerate(tqdm(frames, desc='Segmenting sequence')):
    mask=infer_mask(frame); left,right=split_lungs(mask); visual,lm,rm=visual_frame(frame,left,right,index,fps)
    rows.append({'frame':index,'left_apex_y':lm['apex_y'],'left_bottom_y':lm['bottom_y'],'left_height_px':lm['height_px'],'left_area_px':lm['area_px'],'right_apex_y':rm['apex_y'],'right_bottom_y':rm['bottom_y'],'right_height_px':rm['height_px'],'right_area_px':rm['area_px']})
    if writer is None:
        writer=cv2.VideoWriter(str(mp4_path),cv2.VideoWriter_fourcc(*'mp4v'),fps or 12.0,(visual.shape[1],visual.shape[0]))
        if not writer.isOpened(): raise RuntimeError('Ne mogu da napravim MP4 izlaz.')
    writer.write(visual)
if writer: writer.release()
results=pd.DataFrame(rows)
csv_path=OUTPUT_DIR/f'{sequence_name}_measurements.csv'; results.to_csv(csv_path,index=False)
print('Segmented frames:',len(results))

In [ ]:
# Rezultati testa — sve se prikazuje direktno u notebook-u.
time=results.frame/(fps or 12.0)
window=min(7,len(results) if len(results)%2 else len(results)-1)
smooth=lambda values: savgol_filter(values,window,2) if window>=3 else values
fig,ax=plt.subplots(2,1,figsize=(12,8),sharex=True)
for side,color in (('left','tab:blue'),('right','tab:green')):
    ax[0].plot(time,results[f'{side}_height_px'],color=color,alpha=.25)
    ax[0].plot(time,smooth(results[f'{side}_height_px']),color=color,lw=2,label=f'{side.title()} lung')
    ax[1].plot(time,results[f'{side}_area_px'],color=color,lw=2,label=f'{side.title()} lung')
ax[0].set_ylabel('Height [pixels]'); ax[0].legend(); ax[0].grid(alpha=.2)
ax[1].set(xlabel='Time [s]' if fps else 'Frame',ylabel='Area [pixels squared]'); ax[1].legend(); ax[1].grid(alpha=.2)
fig.tight_layout()
graph_path=OUTPUT_DIR/f'{sequence_name}_graphs.png'; fig.savefig(graph_path,dpi=180); plt.show()
display(results)
# H.264 konverzija omogućava pouzdanu reprodukciju unutar Colab-a.
h264_path=OUTPUT_DIR/f'{sequence_name}_segmentation_h264.mp4'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(mp4_path),'-vcodec','libx264','-pix_fmt','yuv420p',str(h264_path)],check=True)
display(Video(str(h264_path),embed=True,html_attributes='controls width=100%'))
print('CSV:',csv_path)
print('Graph:',graph_path)
print('Video:',h264_path)

In [ ]:
# Opcionalno: preuzmi rezultate lokalno.
from google.colab import files
files.download(str(csv_path))
files.download(str(graph_path))
files.download(str(h264_path))